# Stage 04c: Model Building — Forward Stepwise Selection

Logistic regression model built using forward stepwise variable selection (`pdt.step_fwd()`).

**Dataset:** `runs/2026-03-15_201852/data/loans_binned.csv`  
**Target:** `Creditability` (1 = default)  
**Selection method:** Forward stepwise (p-value based)

In [ ]:
import sys
import os
import json
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, 'c:/projects/superagent/src')
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_curve

import pdtoolkit as pdt

os.chdir('c:/projects/superagent')
RUN_DIR = 'runs/2026-03-15_201852'
FIGURES_DIR = os.path.join(RUN_DIR, 'figures')
PIPELINE_DIR = os.path.join(RUN_DIR, 'pipeline')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(PIPELINE_DIR, exist_ok=True)

print("Setup complete.")

## 1. Load Data and Approved Shortlist

In [ ]:
# Load binned dataset
db_binned = pd.read_csv(os.path.join(RUN_DIR, 'data', 'loans_binned.csv'))
target = 'Creditability'

# Convert target to int 0/1
db_binned[target] = db_binned[target].astype(int)

# Approved shortlist from stage 03
shortlist = [
    'Account Balance',
    'Payment Status of Previous Credit',
    'Duration of Credit (month)',
    'Credit Amount',
    'Value Savings/Stocks',
    'Purpose',
    'Age (years)',
    'Most valuable available asset'
]

# Subset to shortlist + target
db_model = db_binned[shortlist + [target]].copy()

# Ensure all predictor columns are string type for WoE encoding
for col in shortlist:
    db_model[col] = db_model[col].astype(str)

print(f"Dataset: {db_model.shape[0]} rows, {db_model.shape[1]} columns")
print(f"Default rate: {db_model[target].mean():.4f}")
print(f"Shortlist: {shortlist}")

## 2. WoE Encoding

In [ ]:
# WoE encode the shortlist variables using replace_woe
db_woe_input = db_model[shortlist + [target]].copy()
for col in shortlist:
    db_woe_input[col] = db_woe_input[col].astype(str)

db_woe, woe_info = pdt.replace_woe(db_woe_input, target=target)

print("WoE encoding complete.")
if len(woe_info) > 0:
    print(f"WoE info:\n{woe_info}")

# Extract WoE mappings from bivariate tables for each shortlist variable
woe_mappings = {}
for var in shortlist:
    woe_table = pdt.woe_tbl(db_model, x=var, y=target, y_check=False)
    mapping = []
    for _, row in woe_table.iterrows():
        mapping.append({"bin": str(row['bin']), "woe": float(row['woe'])})
    woe_mappings[var] = mapping
    print(f"\n{var} WoE mapping:")
    print(woe_table[['bin', 'no', 'dr', 'woe', 'iv_b']].to_string(index=False))

## 3. Forward Stepwise Variable Selection

In [ ]:
# Run forward stepwise selection on WoE-encoded data
# Note: coding='dummy' is used because step_fwd's WoE mode enforces coef > 0,
# but with target=1=default the WoE coefficients are correctly negative.
# We validate coefficient signs separately in self-assessment check 2.
p_value_threshold = 0.05

fwd_result = pdt.step_fwd(
    start_model="Creditability ~ 1",
    p_value=p_value_threshold,
    coding="dummy",
    db=db_woe,
    risk_factors=shortlist,
    target="Creditability"
)

print("Forward Stepwise Selection Complete")
print(f"\nSteps table:")
print(fwd_result.steps.to_string())
print(f"\nWarnings: {fwd_result.warnings}")

# Extract selected variables from the model summary (exclude const)
selected_vars = [v for v in fwd_result.model['variable'].tolist() if v != 'const']
excluded_vars = [v for v in shortlist if v not in selected_vars]

print(f"\nSelected variables ({len(selected_vars)}): {selected_vars}")
print(f"Excluded variables ({len(excluded_vars)}): {excluded_vars}")

## 4. Self-Assessment Check 1: Variable Count

In [ ]:
# Self-assessment check 1: Variable count
n_vars = len(selected_vars)
var_count_flag = None

if n_vars < 4:
    var_count_flag = f"Too few variables ({n_vars}). Relaxing p-value from {p_value_threshold} to 0.10"
    p_value_relaxed = 0.10
    print(f"WARNING: {var_count_flag}")
    print(f"Re-running forward stepwise with p-value = {p_value_relaxed}")
    
    fwd_result = pdt.step_fwd(
        start_model="Creditability ~ 1",
        p_value=p_value_relaxed,
        coding="dummy",
        db=db_woe,
        risk_factors=shortlist,
        target="Creditability"
    )
    p_value_threshold = p_value_relaxed
    selected_vars = [v for v in fwd_result.model['variable'].tolist() if v != 'const']
    excluded_vars = [v for v in shortlist if v not in selected_vars]
    n_vars = len(selected_vars)
    print(f"After relaxation: {n_vars} variables selected: {selected_vars}")

elif n_vars > 12:
    var_count_flag = f"Too many variables ({n_vars}). Tightening p-value from {p_value_threshold} to 0.01"
    p_value_tight = 0.01
    print(f"WARNING: {var_count_flag}")
    print(f"Re-running forward stepwise with p-value = {p_value_tight}")
    
    fwd_result = pdt.step_fwd(
        start_model="Creditability ~ 1",
        p_value=p_value_tight,
        coding="dummy",
        db=db_woe,
        risk_factors=shortlist,
        target="Creditability"
    )
    p_value_threshold = p_value_tight
    selected_vars = [v for v in fwd_result.model['variable'].tolist() if v != 'const']
    excluded_vars = [v for v in shortlist if v not in selected_vars]
    n_vars = len(selected_vars)
    print(f"After tightening: {n_vars} variables selected: {selected_vars}")

else:
    print(f"Check 1 - Variable count: PASS ({n_vars} variables, in range [4, 12])")

## 5. Final Model Fit

In [ ]:
# Fit final logistic regression with statsmodels on WoE-encoded selected variables
X_woe = db_woe[selected_vars].astype(float)
y = db_woe[target].astype(int)

X_const = sm.add_constant(X_woe)
logit_model = sm.Logit(y, X_const).fit(disp=0)

print(logit_model.summary())
print(f"\nConverged: {logit_model.mle_retvals['converged']}")

## 6. Self-Assessment Checks 2-6

In [ ]:
# --- Self-Assessment Results ---
flags = []
sa_results = {}

# Check 2: Coefficient signs — WoE coefficient should be negative for target=1=default
# In WoE coding: higher WoE = more bads relative to goods
# The logit coefficient on WoE should be negative (lower WoE = lower default probability)
coefficients = logit_model.params.drop('const')
sign_issues = []
for var in selected_vars:
    coef = coefficients[var]
    if coef > 0:
        sign_issues.append(f"{var} (coef={coef:.4f}, expected negative for WoE coding)")

sa_results['coefficient_signs'] = {
    'pass': len(sign_issues) == 0,
    'details': sign_issues if sign_issues else 'All coefficients negative (consistent with WoE coding where target=1=default)'
}
if sign_issues:
    flags.append(f"Coefficient sign reversal: {'; '.join(sign_issues)}")
print("Check 2 - Coefficient signs:", "PASS" if len(sign_issues) == 0 else "FLAG")
for issue in sign_issues:
    print(f"  WARNING: {issue}")

# Check 3: VIF
vif_data = pd.DataFrame()
vif_data['Variable'] = selected_vars
vif_values = []
for i, var in enumerate(selected_vars):
    vif_val = variance_inflation_factor(X_woe.values, i)
    vif_values.append(vif_val)
vif_data['VIF'] = vif_values
vif_flag = vif_data[vif_data['VIF'] > 5]

sa_results['vif'] = {
    'pass': len(vif_flag) == 0,
    'details': vif_data.to_dict('records'),
    'max_vif': max(vif_values)
}
if len(vif_flag) > 0:
    flags.append(f"VIF > 5 for: {', '.join(vif_flag['Variable'].tolist())}")
print(f"\nCheck 3 - VIF: {'PASS' if len(vif_flag) == 0 else 'FLAG'}")
print(vif_data.to_string(index=False))

# Generate predicted probabilities and scores
pred_probs = logit_model.predict(X_const)
scores = pdt.scaled_score(pred_probs, score=600, odd=50, pdo=20)

# Check 4: Score distribution
score_min, score_max = scores.min(), scores.max()
score_mean, score_std = scores.mean(), scores.std()
pct_below_400 = (scores < 400).mean() * 100
pct_above_800 = (scores > 800).mean() * 100
pct_extremes = pct_below_400 + pct_above_800

score_range_ok = (score_min >= 350) and (score_max <= 850)
extremes_ok = pct_extremes <= 5.0

sa_results['score_distribution'] = {
    'pass': score_range_ok and extremes_ok,
    'min': float(score_min), 'max': float(score_max),
    'mean': float(score_mean), 'std': float(score_std),
    'pct_below_400': float(pct_below_400),
    'pct_above_800': float(pct_above_800)
}
if not score_range_ok:
    flags.append(f"Score range [{score_min:.0f}, {score_max:.0f}] outside target [400, 800]")
if not extremes_ok:
    flags.append(f"Extreme scores: {pct_extremes:.1f}% outside [400, 800] (threshold: 5%)")
print(f"\nCheck 4 - Score distribution: {'PASS' if score_range_ok and extremes_ok else 'FLAG'}")
print(f"  Range: [{score_min:.1f}, {score_max:.1f}], Mean: {score_mean:.1f}, Std: {score_std:.1f}")
print(f"  Below 400: {pct_below_400:.1f}%, Above 800: {pct_above_800:.1f}%")

# Check 5: Decile monotonicity
score_df = pd.DataFrame({'score': scores, 'default': y.values})
score_df['decile'] = pd.qcut(score_df['score'], 10, labels=False, duplicates='drop') + 1
decile_stats = score_df.groupby('decile').agg(
    n=('default', 'count'),
    n_defaults=('default', 'sum'),
    default_rate=('default', 'mean'),
    avg_score=('score', 'mean')
).reset_index()

# Check monotonicity: as score increases (higher decile), default rate should decrease
dr_values = decile_stats['default_rate'].values
monotonic_decreasing = all(dr_values[i] >= dr_values[i+1] for i in range(len(dr_values)-1))
n_violations = sum(1 for i in range(len(dr_values)-1) if dr_values[i] < dr_values[i+1])
monotonicity_pass = n_violations <= 1

sa_results['decile_monotonicity'] = {
    'pass': monotonicity_pass,
    'strictly_monotonic': monotonic_decreasing,
    'violations': int(n_violations),
    'details': decile_stats.to_dict('records')
}
if not monotonicity_pass:
    flags.append(f"Decile monotonicity failed: {n_violations} violations")
print(f"\nCheck 5 - Decile monotonicity: {'PASS' if monotonicity_pass else 'FLAG'}")
print(f"  Strictly monotonic: {monotonic_decreasing}, Violations: {n_violations}")
print(decile_stats.to_string(index=False))

## 7. Model Performance Metrics

In [ ]:
# Model performance metrics
model_auc = pdt.auc_model(pred_probs.values, y.values)
model_gini = 2 * model_auc - 1

# KS statistic
fpr, tpr, thresholds = roc_curve(y.values, pred_probs.values)
model_ks = max(tpr - fpr)

# Confusion matrix at optimal cutoff (default rate = 0.30)
cm_result = pdt.confusion_matrix(pred_probs.values, y.values, cutoff=0.3)

print(f"Development AUC:  {model_auc:.4f}")
print(f"Development Gini: {model_gini:.4f}")
print(f"Development KS:   {model_ks:.4f}")
print(f"\nAUC benchmark: {'Strong (>0.80)' if model_auc > 0.80 else 'Acceptable (0.70-0.80)' if model_auc > 0.70 else 'Weak (0.60-0.70)' if model_auc > 0.60 else 'Unacceptable (<0.60)'}")
print(f"Gini benchmark: {'Acceptable (>0.35)' if model_gini > 0.35 else 'Below threshold (<0.35)'}")
print(f"\nConfusion Matrix (cutoff=0.30):")
print(cm_result.confusion_matrix)
print(f"\nClassification Metrics:")
print(cm_result.metrics.to_string())

## 8. Cross-Validation and Bootstrap

In [ ]:
# Cross-validation (10-fold)
cv_db = db_woe[selected_vars + [target]].copy()
for col in selected_vars:
    cv_db[col] = cv_db[col].astype(float)
cv_db[target] = cv_db[target].astype(int)

cv_result = pdt.kfold_vld(
    model=logit_model, db=cv_db, target=target,
    predictors=selected_vars, k=10, seed=1984
)
cv_auc = cv_result.summary['auc'].values[0]

print("10-Fold Cross-Validation Results:")
print(cv_result.iter.to_string(index=False))
print(f"\nCV Summary: AUC = {cv_auc:.4f}")

# Bootstrap validation (500 iterations for speed)
boots_result = pdt.boots_vld(
    model=logit_model, db=cv_db, target=target,
    predictors=selected_vars, B=500, seed=1122
)
boots_auc = boots_result.summary['auc'].values[0]
print(f"\nBootstrap Summary (B=500): AUC = {boots_auc:.4f}")

# Check 6: CV stability
cv_gap = abs(model_auc - cv_auc)
cv_stable = cv_gap <= 0.03
sa_results['cv_stability'] = {
    'pass': cv_stable,
    'dev_auc': float(model_auc),
    'cv_auc': float(cv_auc),
    'boots_auc': float(boots_auc),
    'gap': float(cv_gap)
}
if not cv_stable:
    flags.append(f"CV instability: dev AUC={model_auc:.4f}, CV AUC={cv_auc:.4f}, gap={cv_gap:.4f} > 0.03")
print(f"\nCheck 6 - CV stability: {'PASS' if cv_stable else 'FLAG'}")
print(f"  Dev AUC: {model_auc:.4f}, CV AUC: {cv_auc:.4f}, Gap: {cv_gap:.4f}")
print(f"  Bootstrap AUC: {boots_auc:.4f}")

## 9. Plots

In [ ]:
# --- Plot 1: ROC Curve ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, color='darkblue', lw=2, label=f'FWD Model (AUC = {model_auc:.4f})')
ax.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve \u2014 Forward Stepwise Model', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '04c_roc_curve.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: 04c_roc_curve.png")

In [ ]:
# --- Plot 2: Score Distribution ---
fig, ax = plt.subplots(figsize=(10, 6))
scores_good = scores[y.values == 0]
scores_bad = scores[y.values == 1]
ax.hist(scores_good, bins=30, alpha=0.6, color='steelblue', label=f'Good (n={len(scores_good)})', density=True)
ax.hist(scores_bad, bins=30, alpha=0.6, color='salmon', label=f'Bad (n={len(scores_bad)})', density=True)
ax.axvline(x=score_mean, color='black', linestyle='--', lw=1.5, label=f'Mean = {score_mean:.0f}')
ax.set_xlabel('Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Score Distribution \u2014 Forward Stepwise Model', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '04c_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: 04c_score_distribution.png")

In [ ]:
# --- Plot 3: Score Decile Table ---
fig, ax1 = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(decile_stats))
bars = ax1.bar(x_pos, decile_stats['n'], color='steelblue', alpha=0.6, label='Count')
ax1.set_xlabel('Score Decile', fontsize=12)
ax1.set_ylabel('Count', fontsize=12, color='steelblue')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(decile_stats['decile'].values)

ax2 = ax1.twinx()
ax2.plot(x_pos, decile_stats['default_rate'] * 100, color='red', marker='o', lw=2, label='Default Rate (%)')
ax2.set_ylabel('Default Rate (%)', fontsize=12, color='red')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=11)

ax1.set_title('Score Decile Analysis \u2014 Forward Stepwise Model', fontsize=14)
ax1.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '04c_score_decile_table.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: 04c_score_decile_table.png")

In [ ]:
# --- Plot 4: Coefficient Plot ---
coef_df = pd.DataFrame({
    'Variable': selected_vars,
    'Coefficient': [coefficients[v] for v in selected_vars]
}).sort_values('Coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
ax.barh(coef_df['Variable'], coef_df['Coefficient'], color=colors, alpha=0.7)
ax.axvline(x=0, color='black', lw=0.8)
ax.set_xlabel('Coefficient', fontsize=12)
ax.set_title('Logistic Regression Coefficients \u2014 Forward Stepwise Model', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, '04c_coefficient_plot.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved: 04c_coefficient_plot.png")

## 10. Save Model Parameters and Stage Summary

In [ ]:
# Build model_params_fwd.json
# Only include WoE mappings for selected variables
selected_woe_mappings = {var: woe_mappings[var] for var in selected_vars if var in woe_mappings}

model_params = {
    "selection_method": "forward_stepwise",
    "selected_variables": selected_vars,
    "woe_mappings": selected_woe_mappings,
    "coefficients": {var: float(coefficients[var]) for var in selected_vars},
    "intercept": float(logit_model.params['const']),
    "score_params": {"base_score": 600, "base_odds": 50, "pdo": 20},
    "model_auc": float(model_auc),
    "model_gini": float(model_gini),
    "model_ks": float(model_ks)
}

params_path = os.path.join(PIPELINE_DIR, 'model_params_fwd.json')
with open(params_path, 'w') as f:
    json.dump(model_params, f, indent=2)
print(f"Saved: {params_path}")

In [ ]:
# Build stage_04c.md
summary_model = logit_model.summary2().tables[0]

# Build coefficients table
coef_table_data = []
for var in ['const'] + selected_vars:
    coef_table_data.append({
        'Variable': var,
        'Coef': f"{logit_model.params[var]:.4f}",
        'Std Err': f"{logit_model.bse[var]:.4f}",
        'z': f"{logit_model.tvalues[var]:.4f}",
        'P>|z|': f"{logit_model.pvalues[var]:.4f}",
        '[0.025': f"{logit_model.conf_int().loc[var, 0]:.4f}",
        '0.975]': f"{logit_model.conf_int().loc[var, 1]:.4f}"
    })
coef_table = pd.DataFrame(coef_table_data)

# Build forward stepwise steps table
steps_table = ""
if len(fwd_result.steps) > 0:
    for _, row in fwd_result.steps.iterrows():
        steps_table += f"| {int(row['iteration'])} | {row['rf_added']} | {row['p_value']:.4f} | {row['aic']:.2f} |\n"

# Exclusion reasons
exclusion_reasons = []
for var in excluded_vars:
    exclusion_reasons.append(f"{var}: p-value above threshold ({p_value_threshold})")

stage_md = f"""selection_method: forward_stepwise
binned_dataset_path: {RUN_DIR}/data/loans_binned.csv
selected_variables: {selected_vars}
excluded_from_shortlist: {exclusion_reasons}
model_auc: {model_auc:.4f}
model_gini: {model_gini:.4f}
model_ks: {model_ks:.4f}
p_value_threshold_used: {p_value_threshold}
cv_auc: {cv_auc:.4f}
bootstrap_auc: {boots_auc:.4f}

## Forward Stepwise Selection Steps

| Step | Variable Added | LR p-value | AIC |
|------|---------------|------------|-----|
{steps_table}

## Logistic Regression Model Fit

| Metric | Value |
|---|---|
| Dep. Variable | {target} |
| No. Observations | {int(logit_model.nobs)} |
| Df Model | {int(logit_model.df_model)} |
| Pseudo R-squared | {logit_model.prsquared:.4f} |
| Log-Likelihood | {logit_model.llf:.2f} |
| LL-Null | {logit_model.llnull:.2f} |
| LLR p-value | {logit_model.llr_pvalue:.2e} |
| Converged | {logit_model.mle_retvals['converged']} |

## Coefficients

{coef_table.to_markdown(index=False)}

## Self-Assessment Results

### Check 1: Variable Count
- Variables selected: {n_vars}
- Status: {"PASS" if 4 <= n_vars <= 12 else "ADJUSTED"}
{f"- Note: {var_count_flag}" if var_count_flag else ""}

### Check 2: Coefficient Signs
- All negative (consistent with WoE coding): {sa_results['coefficient_signs']['pass']}
- Details: {sa_results['coefficient_signs']['details']}

### Check 3: VIF (Multicollinearity)
- All VIF < 5: {sa_results['vif']['pass']}
- Max VIF: {sa_results['vif']['max_vif']:.2f}

### Check 4: Score Distribution
- Range: [{sa_results['score_distribution']['min']:.1f}, {sa_results['score_distribution']['max']:.1f}]
- Mean: {sa_results['score_distribution']['mean']:.1f}, Std: {sa_results['score_distribution']['std']:.1f}
- Below 400: {sa_results['score_distribution']['pct_below_400']:.1f}%, Above 800: {sa_results['score_distribution']['pct_above_800']:.1f}%

### Check 5: Decile Monotonicity
- Strictly monotonic: {sa_results['decile_monotonicity']['strictly_monotonic']}
- Violations: {sa_results['decile_monotonicity']['violations']}

### Check 6: CV Stability
- Dev AUC: {model_auc:.4f}, CV AUC: {cv_auc:.4f}, Gap: {cv_gap:.4f}
- Bootstrap AUC: {boots_auc:.4f}
- Stable (gap <= 0.03): {sa_results['cv_stability']['pass']}

## Flags
{chr(10).join(f"- {f}" for f in flags) if flags else "None"}
"""

stage_path = os.path.join(PIPELINE_DIR, 'stage_04c.md')
with open(stage_path, 'w') as f:
    f.write(stage_md)
print(f"Saved: {stage_path}")

In [ ]:
# --- Fix-Proposer Smoke Tests ---
fix_issues = []

# Test 1: Coefficient sign consistency
if not sa_results['coefficient_signs']['pass']:
    fix_issues.append({
        'test': 'Coefficient sign consistency',
        'severity': 'critical',
        'detail': f"Sign reversals detected: {sa_results['coefficient_signs']['details']}"
    })

# Test 2: Multicollinearity
if sa_results['vif']['max_vif'] >= 10:
    fix_issues.append({
        'test': 'Multicollinearity (VIF)',
        'severity': 'critical',
        'detail': f"VIF >= 10 detected (max: {sa_results['vif']['max_vif']:.2f})"
    })
elif sa_results['vif']['max_vif'] >= 5:
    fix_issues.append({
        'test': 'Multicollinearity (VIF)',
        'severity': 'warning',
        'detail': f"VIF >= 5 detected (max: {sa_results['vif']['max_vif']:.2f})"
    })

# Test 3: Minimum discrimination
if model_auc < 0.60:
    fix_issues.append({
        'test': 'Minimum discrimination',
        'severity': 'critical',
        'detail': f"AUC = {model_auc:.4f} < 0.60 (unacceptable)"
    })
elif model_auc < 0.70:
    fix_issues.append({
        'test': 'Minimum discrimination',
        'severity': 'warning',
        'detail': f"AUC = {model_auc:.4f} < 0.70 (weak)"
    })

# Test 4: Score range
if score_min > 800 or score_max < 400:
    fix_issues.append({
        'test': 'Score range',
        'severity': 'critical',
        'detail': f"Score range [{score_min:.0f}, {score_max:.0f}] does not overlap with [400, 800]"
    })

# Test 5: Model params completeness
required_keys = ['selection_method', 'selected_variables', 'woe_mappings', 'coefficients',
                 'intercept', 'score_params', 'model_auc', 'model_gini', 'model_ks']
missing_keys = [k for k in required_keys if k not in model_params]
if missing_keys:
    fix_issues.append({
        'test': 'Model params completeness',
        'severity': 'critical',
        'detail': f"Missing keys: {missing_keys}"
    })

# Write fixes file
n_critical = sum(1 for i in fix_issues if i['severity'] == 'critical')
n_warning = sum(1 for i in fix_issues if i['severity'] == 'warning')

fixes_md = f"""# Stage 04c Fix-Proposer Diagnostic

## Summary
- Total issues: {len(fix_issues)}
- Critical: {n_critical}
- Warning: {n_warning}

## Smoke Tests

| # | Test | Severity | Result |
|---|------|----------|--------|
| 1 | Coefficient sign consistency | - | {'FAIL' if any(i['test'] == 'Coefficient sign consistency' for i in fix_issues) else 'PASS'} |
| 2 | Multicollinearity (VIF < 10) | - | {'FAIL' if any(i['test'] == 'Multicollinearity (VIF)' and i['severity'] == 'critical' for i in fix_issues) else 'WARN' if any(i['test'] == 'Multicollinearity (VIF)' for i in fix_issues) else 'PASS'} |
| 3 | Minimum discrimination (AUC > 0.60) | - | {'FAIL' if any(i['test'] == 'Minimum discrimination' and i['severity'] == 'critical' for i in fix_issues) else 'WARN' if any(i['test'] == 'Minimum discrimination' for i in fix_issues) else 'PASS'} |
| 4 | Score range overlaps [400, 800] | - | {'FAIL' if any(i['test'] == 'Score range' for i in fix_issues) else 'PASS'} |
| 5 | Model params complete | - | {'FAIL' if any(i['test'] == 'Model params completeness' for i in fix_issues) else 'PASS'} |

## Details
"""
for issue in fix_issues:
    fixes_md += f"\n### [{issue['severity'].upper()}] {issue['test']}\n{issue['detail']}\n"

if not fix_issues:
    fixes_md += "\nAll smoke tests passed. No issues detected.\n"

fixes_path = os.path.join(PIPELINE_DIR, 'stage_04c_fixes.md')
with open(fixes_path, 'w') as f:
    f.write(fixes_md)
print(f"Saved: {fixes_path}")
print(f"\nFix-proposer: {len(fix_issues)} issues ({n_critical} critical, {n_warning} warning)")

In [ ]:
# --- Final Summary ---
print("=" * 70)
print("STAGE 04c (Forward Stepwise) \u2014 COMPLETE")
print("=" * 70)
print(f"Variables selected: {n_vars}")
print(f"  {selected_vars}")
print(f"Development AUC:  {model_auc:.4f}")
print(f"Development Gini: {model_gini:.4f}")
print(f"Development KS:   {model_ks:.4f}")
print(f"CV AUC:           {cv_auc:.4f}")
print(f"Bootstrap AUC:    {boots_auc:.4f}")
print(f"Score range:      [{score_min:.0f}, {score_max:.0f}], mean {score_mean:.0f}")
print(f"All coef signs consistent: {sa_results['coefficient_signs']['pass']}")
print(f"Decile monotonicity: {'pass' if monotonicity_pass else 'fail'}")
print(f"Flags: {flags if flags else 'None'}")
print(f"Fix proposals: {len(fix_issues)} issues ({n_critical} critical)")
print(f"\nOutputs:")
print(f"  Notebook:     {RUN_DIR}/notebooks/04c_model_building_fwd.ipynb")
print(f"  Model params: {PIPELINE_DIR}/model_params_fwd.json")
print(f"  Stage summary: {PIPELINE_DIR}/stage_04c.md")
print(f"  Fix proposals: {PIPELINE_DIR}/stage_04c_fixes.md")